# 🏁 SOTA v28 Ultimate Classifier Chain Pipeline (End-to-End)

This notebook implements the complete end-to-end SOTA v28 pipeline. 
It integrates data pre-processing, 6-model sequential classifier chain training, out-of-fold subject-wise Beta Calibration, SLSQP Stacking, and final Personalized Calibration.

### Pipeline Flow:
1. **Core Imports & Environment Settings**
2. **Leak-Free Transformers & Validation Splits**
3. **Chronological Forward-Only Target Feature Propagation**
4. **Robust Directives (Pairwise Interaction Generation)**
5. **Stage 1: 6 GBDT & Linear Base Models Classifier Chain Training**
6. **Stage 2: Out-of-Fold Beta Calibration & SLSQP Stacking Blender**
7. **Stage 3: Personalized Baseline Calibration & Final Submission Compilation**

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

# Suppress warnings
warnings.filterwarnings('ignore')

# Ensure project root is in sys.path
project_root = Path(os.getcwd()).resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data.data_loader import DataLoader
from src.data.make_dataset import build_training_dataset
from src.features.build_features import engineer_features
from src.features.transformers import FoldWisePCA, LeakFreeSubjectTargetEncoder, LeakFreeSubjectZScaler
from src.utils.logger import get_logger
from src.utils.validation import make_subject_hole_folds

logger = get_logger(__name__)
print("✅ All core SOTA-v28 libraries successfully imported!")

In [ ]:
def add_temporal_target_features(df, y, train_idx, target_col, id_col, half_life=3.0):
    df_copy = df.copy()
    df_copy['original_index'] = df_copy.index

    # Isolate training labels
    y_train = pd.Series(np.nan, index=df_copy.index)
    y_train.loc[train_idx] = y.loc[train_idx]
    df_copy['y_train_temp'] = y_train

    df_copy['date_temp'] = pd.to_datetime(df_copy['date'])
    df_copy = df_copy.sort_values(by=[id_col, 'date_temp']).reset_index(drop=True)

    # Shift to prevent target leakage
    df_copy['date_with_target'] = df_copy['date_temp'].where(df_copy['y_train_temp'].notna())
    df_copy['y_train_prev_source'] = df_copy.groupby(id_col)['y_train_temp'].shift(1)
    df_copy['date_prev_source'] = df_copy.groupby(id_col)['date_with_target'].shift(1)

    # STRICT LEAK-FREE TARGET FLOW: Future bfill() completely removed to prevent Look-Ahead target leakage!
    df_copy['y_prev'] = df_copy.groupby(id_col)['y_train_prev_source'].ffill()
    df_copy['date_prev'] = df_copy.groupby(id_col)['date_prev_source'].ffill()

    df_copy['delta_prev'] = (df_copy['date_temp'] - pd.to_datetime(df_copy['date_prev'])).dt.days

    # Calculate subject-specific mean baseline (Laplace smoothed)
    subject_means = df_copy.groupby(id_col)['y_train_temp'].transform('mean').fillna(0.5)

    # Apply Exponential Decay Chronological Weighted Interpolation (Forward-only)
    decay_lambda = np.log(2.0) / half_life
    w_prev = np.exp(-decay_lambda * df_copy['delta_prev'].fillna(999.0))

    prior_weight = 0.2
    denom = w_prev + prior_weight
    df_copy['y_interp'] = (w_prev * df_copy['y_prev'].fillna(0.5) + prior_weight * subject_means) / denom

    # Next/Future targets are forced to subject_means to maintain schema compatibility without any look-ahead leak!
    df_copy['y_prev'] = df_copy['y_prev'].fillna(subject_means)
    df_copy['y_next'] = subject_means
    df_copy['y_interp'] = df_copy['y_interp'].fillna(subject_means)
    df_copy['delta_next'] = 999.0  # Force infinity/no next info

    df_copy = df_copy.sort_values(by='original_index').reset_index(drop=True)

    features = pd.DataFrame(index=df.index)
    features[f'{target_col}_prev'] = df_copy['y_prev'].values
    features[f'{target_col}_next'] = df_copy['y_next'].values
    features[f'{target_col}_interp'] = df_copy['y_interp'].values
    features[f'{target_col}_log_delta_prev'] = np.log1p(df_copy['delta_prev'].fillna(999.0)).values
    features[f'{target_col}_log_delta_next'] = np.log1p(df_copy['delta_next']).values

    return features

print("✅ Temporal target feature function defined.")

In [ ]:
def apply_robust_directives(df_train_fold, df_test_fold, tr_idx, target, id_col, numeric_cols, y_train_fold):
    df_train_fold = df_train_fold.copy()
    df_test_fold = df_test_fold.copy()

    special_cols = []

    if target in ["S2", "S3", "S4"]:
        candidate_cols = [c for c in df_train_fold.columns if c.endswith("_subject_zscore")]
        corrs = {}
        for col in candidate_cols:
            c = df_train_fold.iloc[tr_idx][col].corr(y_train_fold.iloc[tr_idx])
            if pd.notna(c):
                corrs[col] = abs(c)
        top_5 = sorted(corrs.items(), key=lambda x: x[1], reverse=True)[:5]
        top_5_cols = [col for col, _ in top_5]

        for i in range(len(top_5_cols)):
            for j in range(i + 1, len(top_5_cols)):
                c0, c1 = top_5_cols[i], top_5_cols[j]

                # Add
                name_add = f"{c0}_plus_{c1}"
                df_train_fold[name_add] = df_train_fold[c0] + df_train_fold[c1]
                df_test_fold[name_add] = df_test_fold[c0] + df_test_fold[c1]
                special_cols.append(name_add)

                # Subtract
                name_sub = f"{c0}_minus_{c1}"
                df_train_fold[name_sub] = df_train_fold[c0] - df_train_fold[c1]
                df_test_fold[name_sub] = df_test_fold[c0] - df_test_fold[c1]
                special_cols.append(name_sub)

                # Multiply
                name_mul = f"{c0}_mul_{c1}"
                df_train_fold[name_mul] = df_train_fold[c0] * df_train_fold[c1]
                df_test_fold[name_mul] = df_test_fold[c0] * df_test_fold[c1]
                special_cols.append(name_mul)

    return df_train_fold, df_test_fold, special_cols

print("✅ Robust directives (interaction generation) function defined.")

In [ ]:
print("🔄 Stage 1: Initializing data load & SOTA base feature generation...")
data_loader = DataLoader(data_dir="data")
metrics_df = data_loader.load_metrics()
sample_sub_df = data_loader.load_sample_submission()

processed_train_path = Path("data/processed/df_train_raw.parquet")
processed_test_path = Path("data/processed/df_test_raw.parquet")

if processed_train_path.exists() and processed_test_path.exists():
    print("Found cached raw datasets in data/processed/. Loading directly...")
    df_train_raw = pd.read_parquet(processed_train_path)
    df_test_raw = pd.read_parquet(processed_test_path)
else:
    print("No cached datasets found. Rebuilding from raw sensor parquets...")
    sensor_data_dict = data_loader.load_sensor_data()
    df_train_raw = build_training_dataset(metrics_df, sensor_data_dict)
    df_test_raw = build_training_dataset(sample_sub_df, sensor_data_dict)

    # Save to cache
    processed_train_path.parent.mkdir(parents=True, exist_ok=True)
    df_train_raw.to_parquet(processed_train_path, index=False)
    df_test_raw.to_parquet(processed_test_path, index=False)
    print("Cached raw datasets to data/processed/")

df_train_raw['is_test'] = 0
df_test_raw['is_test'] = 1
df_all_raw = pd.concat([df_train_raw, df_test_raw], ignore_index=True)

df_all = engineer_features(df_all_raw)

df_train = df_all[df_all['is_test'] == 0].drop(columns=['is_test']).copy()
df_test = df_all[df_all['is_test'] == 1].drop(columns=['is_test']).copy()

id_col = "subject_id" if "subject_id" in df_train.columns else "user_id"
targets = ["Q1", "Q2", "Q3", "S1", "S2", "S3", "S4"]

print(f"Data loaded! Train shape: {df_train.shape}, Test shape: {df_test.shape}")

In [ ]:
lstm_oof_path = 'submissions/oof_dl_v10.csv'
lstm_sub_path = 'submissions/submission_dl_v10.csv'

base_feature_cols = []
if os.path.exists(lstm_oof_path) and os.path.exists(lstm_sub_path):
    print("Deep Learning sequence models found. Merging as anchor stack features...")
    base_oof = pd.read_csv(lstm_oof_path)
    base_oof['date'] = pd.to_datetime(base_oof['date']).dt.date
    df_train['date'] = pd.to_datetime(df_train['date']).dt.date
    df_train = pd.merge(df_train, base_oof, on=[id_col, 'date'], how='left', suffixes=('', '_base'))

    base_sub = pd.read_csv(lstm_sub_path)
    if 'date' not in base_sub.columns:
        if 'lifelog_date' in base_sub.columns:
            base_sub['date'] = base_sub['lifelog_date']
        elif 'sleep_date' in base_sub.columns:
            base_sub['date'] = base_sub['sleep_date']

    base_sub['date'] = pd.to_datetime(base_sub['date']).dt.date
    df_test['date'] = pd.to_datetime(df_test['date']).dt.date
    df_test = pd.merge(df_test, base_sub, on=[id_col, 'date'], how='left', suffixes=('', '_base'))

    eps = 1e-6
    for t in targets:
        base_col = f"{t}_base"
        if base_col in df_train.columns:
            prob = df_train[base_col].fillna(0.5).clip(eps, 1-eps)
            df_train[f'base_{t}_prob'] = prob
            df_train[f'base_{t}_logit'] = np.log(prob / (1 - prob))
        if base_col in df_test.columns:
            prob_t = df_test[base_col].fillna(0.5).clip(eps, 1-eps)
            df_test[f'base_{t}_prob'] = prob_t
            df_test[f'base_{t}_logit'] = np.log(prob_t / (1 - prob_t))
        base_feature_cols.extend([f'base_{t}_prob', f'base_{t}_logit'])
else:
    print("⚠️ No sequence Deep Learning anchor OOFs found. Running without deep features.")

In [ ]:
print("🔄 Preparing validation splits and predictor configurations...")
n_splits = 5
df_train = df_train.sort_values(by=[id_col, 'date']).reset_index(drop=True)
df_test = df_test.sort_values(by=[id_col, 'date']).reset_index(drop=True)

cv_folds = make_subject_hole_folds(df_train, n_folds=n_splits)

drop_cols = targets + [id_col, "date", "timestamp", "sleep_date", "lifelog_date"]
for t in targets:
    drop_cols.extend([f"{t}_zscore", f"{t}_is_missing"])
    if f"{t}_base" in df_train.columns:
        drop_cols.append(f"{t}_base")

feature_cols = [c for c in df_train.columns if c not in drop_cols and c not in base_feature_cols]
numeric_cols = df_train[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

# Base OOF predictions and test projections arrays
oof_preds_cat = np.zeros((len(df_train), len(targets)))
oof_preds_lgb = np.zeros((len(df_train), len(targets)))
oof_preds_xgb = np.zeros((len(df_train), len(targets)))
oof_preds_et = np.zeros((len(df_train), len(targets)))
oof_preds_rf = np.zeros((len(df_train), len(targets)))
oof_preds_lr = np.zeros((len(df_train), len(targets)))

test_preds_cat = np.zeros((len(df_test), len(targets)))
test_preds_lgb = np.zeros((len(df_test), len(targets)))
test_preds_xgb = np.zeros((len(df_test), len(targets)))
test_preds_et = np.zeros((len(df_test), len(targets)))
test_preds_rf = np.zeros((len(df_test), len(targets)))
test_preds_lr = np.zeros((len(df_test), len(targets)))

chain_order = ["Q1", "S1", "S3", "S2", "S4", "Q2", "Q3"]

print(f"Validation setup finished. Selected base GBDT & Linear feature candidates count: {len(numeric_cols)}")

In [ ]:
print("🚀 Starting Stage 1 Base Models Sequential Classifier Chain Training...")

for chain_step, target in enumerate(chain_order):
    t_global_idx = targets.index(target)
    print(f"\n=== [Chain Step {chain_step+1}/7] Target: {target} ===")

    # Inject preceding chain labels dynamically out-of-fold
    prev_targets = chain_order[:chain_step]
    chain_feature_cols = []
    for prev_t in prev_targets:
        prev_global_idx = targets.index(prev_t)

        blend_prev_oof = (1/6.0) * (oof_preds_cat[:, prev_global_idx] +
                                    oof_preds_lgb[:, prev_global_idx] +
                                    oof_preds_xgb[:, prev_global_idx] +
                                    oof_preds_et[:, prev_global_idx] +
                                    oof_preds_rf[:, prev_global_idx] +
                                    oof_preds_lr[:, prev_global_idx])

        blend_prev_test = (1/6.0) * (test_preds_cat[:, prev_global_idx] +
                                     test_preds_lgb[:, prev_global_idx] +
                                     test_preds_xgb[:, prev_global_idx] +
                                     test_preds_et[:, prev_global_idx] +
                                     test_preds_rf[:, prev_global_idx] +
                                     test_preds_lr[:, prev_global_idx])

        df_train[f'meta_prob_{prev_t}'] = blend_prev_oof
        df_test[f'meta_prob_{prev_t}'] = blend_prev_test

        eps = 1e-6
        df_train[f'meta_logit_{prev_t}'] = np.log(df_train[f'meta_prob_{prev_t}'].clip(eps, 1-eps) / (1 - df_train[f'meta_prob_{prev_t}'].clip(eps, 1-eps)))
        df_test[f'meta_logit_{prev_t}'] = np.log(df_test[f'meta_prob_{prev_t}'].clip(eps, 1-eps) / (1 - df_test[f'meta_prob_{prev_t}'].clip(eps, 1-eps)))

        chain_feature_cols.extend([f'meta_prob_{prev_t}', f'meta_logit_{prev_t}'])

    y = df_train[target]
    has_label = y.notna()
    valid_indices = np.where(has_label)[0]
    valid_idx_set = set(valid_indices)

    for fold, (train_idx, val_idx) in enumerate(cv_folds):
        tr_idx = np.array([i for i in train_idx if i in valid_idx_set])
        va_idx = np.array([i for i in val_idx if i in valid_idx_set])

        if len(tr_idx) == 0 or len(va_idx) == 0:
            continue

        df_train_fold = df_train.copy()
        y_train_fold = y.copy()

        # Add temporal exponential decay features (strictly forward-only)
        temp_features = add_temporal_target_features(df_train_fold, y_train_fold, tr_idx, target, id_col)

        temp_test = add_temporal_target_features(
            pd.concat([df_train_fold, df_test], ignore_index=True),
            pd.concat([y_train_fold, pd.Series(index=df_test.index)], ignore_index=True),
            np.arange(len(df_train_fold)),
            target,
            id_col
        )
        temp_features_test = temp_test.iloc[len(df_train_fold):].reset_index(drop=True)

        df_train_fold = pd.concat([df_train_fold, temp_features], axis=1)
        df_test_fold = pd.concat([df_test, temp_features_test], axis=1)

        # Subject Z-scaling
        z_scaler = LeakFreeSubjectZScaler(id_col=id_col, numeric_cols=numeric_cols)
        z_scaler.fit(df_train_fold.iloc[tr_idx])
        df_train_fold = z_scaler.transform(df_train_fold)
        df_test_fold = z_scaler.transform(df_test_fold)

        # Target encoding
        t_encoder = LeakFreeSubjectTargetEncoder(id_col=id_col)
        t_encoder.fit(df_train_fold.iloc[tr_idx], y_train_fold.iloc[tr_idx])
        df_train_fold = t_encoder.transform(df_train_fold)
        df_test_fold = t_encoder.transform(df_test_fold)

        # Interaction Features
        df_train_fold, df_test_fold, special_cols = apply_robust_directives(
            df_train_fold, df_test_fold, tr_idx, target, id_col, numeric_cols, y_train_fold
        )

        # PCA features (for sleep target variables)
        pca_cols = []
        if target in ["S2", "S3", "S4"]:
            pca_transformer = FoldWisePCA(n_components=5, numeric_cols=numeric_cols)
            pca_transformer.fit(df_train_fold.iloc[tr_idx])
            df_train_fold = pca_transformer.transform(df_train_fold)
            df_test_fold = pca_transformer.transform(df_test_fold)
            pca_cols = [f"pca_comp_{idx}" for idx in range(5)]

        # Feature Correlation Filter selection
        stability_scores = {}
        if target in ["S1", "S2", "S3", "S4"]:
            candidate_cols = [c for c in df_train_fold.columns if c not in drop_cols and c not in base_feature_cols and not c.endswith("_freq") and "_plus_" not in c and "_minus_" not in c and "_mul_" not in c] + special_cols
        else:
            candidate_cols = [c for c in df_train_fold.columns if c.endswith("_subject_zscore")] + special_cols

        for feat in candidate_cols:
            corr = df_train_fold.iloc[tr_idx][feat].corr(y_train_fold.iloc[tr_idx])
            if pd.notna(corr):
                stability_scores[feat] = abs(corr)

        selected_candidates = [k for k, v in stability_scores.items() if v >= 0.15]
        if len(selected_candidates) < 30:
            sorted_feats = sorted(stability_scores.items(), key=lambda x: x[1], reverse=True)
            selected_candidates = [f for f, v in sorted_feats[:80] if v >= 0.05]
        else:
            sorted_feats = sorted(stability_scores.items(), key=lambda x: x[1], reverse=True)
            selected_candidates = [f for f, v in sorted_feats[:80] if v >= 0.10]

        temporal_cols = [f'{target}_prev', f'{target}_next', f'{target}_interp', f'{target}_log_delta_prev', f'{target}_log_delta_next']
        active_cols = selected_candidates + base_feature_cols + chain_feature_cols + ["subject_target_encoded"] + pca_cols + temporal_cols + special_cols

        # Unique columns preserving insertion order
        seen = set()
        active_cols = [x for x in active_cols if not (x in seen or seen.add(x))]

        X_tr = df_train_fold.iloc[tr_idx][active_cols].fillna(0.5)
        y_tr = y_train_fold.iloc[tr_idx]
        X_va = df_train_fold.iloc[va_idx][active_cols].fillna(0.5)
        X_te = df_test_fold[active_cols].fillna(0.5)

        # 1. CatBoost Classifier
        cat_model = CatBoostClassifier(iterations=250, depth=3, learning_rate=0.02, l2_leaf_reg=20.0, random_state=42, verbose=0, thread_count=2)
        cat_model.fit(X_tr, y_tr)
        oof_preds_cat[va_idx, t_global_idx] = cat_model.predict_proba(X_va)[:, 1]
        test_preds_cat[:, t_global_idx] += cat_model.predict_proba(X_te)[:, 1] / n_splits

        # 2. LightGBM Classifier
        lgb_model = lgb.LGBMClassifier(n_estimators=150, max_depth=3, learning_rate=0.02, min_child_samples=10, reg_alpha=5.0, reg_lambda=5.0, random_state=42, verbose=-1, n_jobs=2)
        lgb_model.fit(X_tr, y_tr)
        oof_preds_lgb[va_idx, t_global_idx] = lgb_model.predict_proba(X_va)[:, 1]
        test_preds_lgb[:, t_global_idx] += lgb_model.predict_proba(X_te)[:, 1] / n_splits

        # 3. XGBoost Classifier
        xgb_model = xgb.XGBClassifier(n_estimators=150, max_depth=3, learning_rate=0.02, reg_alpha=5.0, reg_lambda=5.0, random_state=42, eval_metric="logloss", n_jobs=2)
        xgb_model.fit(X_tr, y_tr)
        oof_preds_xgb[va_idx, t_global_idx] = xgb_model.predict_proba(X_va)[:, 1]
        test_preds_xgb[:, t_global_idx] += xgb_model.predict_proba(X_te)[:, 1] / n_splits

        # 4. Extra Trees Classifier
        et_model = ExtraTreesClassifier(n_estimators=300, max_depth=6, min_samples_leaf=3, random_state=42, n_jobs=2)
        et_model.fit(X_tr, y_tr)
        oof_preds_et[va_idx, t_global_idx] = et_model.predict_proba(X_va)[:, 1]
        test_preds_et[:, t_global_idx] += et_model.predict_proba(X_te)[:, 1] / n_splits

        # 5. Random Forest Classifier
        rf_model = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=3, random_state=42, n_jobs=2)
        rf_model.fit(X_tr, y_tr)
        oof_preds_rf[va_idx, t_global_idx] = rf_model.predict_proba(X_va)[:, 1]
        test_preds_rf[:, t_global_idx] += rf_model.predict_proba(X_te)[:, 1] / n_splits

        # 6. Logistic Regression Classifier
        lr_model = LogisticRegression(C=0.1, penalty='l2', solver='lbfgs', random_state=42)
        lr_model.fit(X_tr, y_tr)
        oof_preds_lr[va_idx, t_global_idx] = lr_model.predict_proba(X_va)[:, 1]
        test_preds_lr[:, t_global_idx] += lr_model.predict_proba(X_te)[:, 1] / n_splits

    print(f"Target {target} Base CV Log-Loss:")
    print(f"  - CatBoost: {log_loss(y[has_label], oof_preds_cat[has_label, t_global_idx]):.5f}")
    print(f"  - LightGBM: {log_loss(y[has_label], oof_preds_lgb[has_label, t_global_idx]):.5f}")
    print(f"  - ExtraTrees: {log_loss(y[has_label], oof_preds_et[has_label, t_global_idx]):.5f}")
    print(f"  - LogisticReg: {log_loss(y[has_label], oof_preds_lr[has_label, t_global_idx]):.5f}")

## 📈 Stage 2 & 3: SLSQP Stacking Blender & Beta/Personalized Calibration

This section optimizes model blending weights using the SLSQP algorithm, performs out-of-fold subject-wise beta calibration, and applies personalized subject baseline targets to ensure 0% leakage with optimal test projection accuracy.

In [ ]:
def calibrate_beta_subject_wise(y_true, p_raw, subjects, global_T=1.0):
    eps = 1e-15
    p_raw = np.clip(p_raw, eps, 1 - eps)
    logits = np.log(p_raw / (1 - p_raw))

    unique_subjects = np.unique(subjects)
    calibrated_p = np.zeros_like(p_raw)

    for subject in unique_subjects:
        subj_mask = subjects == subject
        other_mask = ~subj_mask

        if other_mask.sum() > 20:
            clf = LogisticRegression(C=0.1, penalty='l2', solver='lbfgs', random_state=42)
            clf.fit(logits[other_mask].reshape(-1, 1), y_true[other_mask])
            calibrated_p[subj_mask] = clf.predict_proba(logits[subj_mask].reshape(-1, 1))[:, 1]
        else:
            calibrated_p[subj_mask] = 1 / (1 + np.exp(-logits[subj_mask] / global_T))

    return np.clip(calibrated_p, eps, 1 - eps)

def calibrate_beta_global(y_true, p_raw):
    eps = 1e-15
    p_raw = np.clip(p_raw, eps, 1 - eps)
    logits = np.log(p_raw / (1 - p_raw))

    clf = LogisticRegression(C=0.1, penalty='l2', solver='lbfgs', random_state=42)
    clf.fit(logits.reshape(-1, 1), y_true)
    return clf

print("✅ Calibration helper functions defined.")

In [ ]:
print("🔄 Preparing files and OOF predictors for meta blending...")
os.makedirs("submissions", exist_ok=True)

oof_template = pd.DataFrame({'date': df_train['date'], 'subject_id': df_train[id_col]})
sub_template = pd.DataFrame({'date': df_test['date'], 'subject_id': df_test[id_col]})

raw_oof_preds = [
    oof_preds_cat,
    oof_preds_lgb,
    oof_preds_xgb,
    oof_preds_et,
    oof_preds_rf,
    oof_preds_lr
]

sub_dfs = []
for idx, name in enumerate(['catboost', 'lgbm', 'xgb', 'et', 'rf', 'lr']):
    oof_df = oof_template.copy()
    oof_df[targets] = raw_oof_preds[idx]
    oof_df.to_csv(f"submissions/oof_{name}_v10.csv", index=False)

    sub_df = sub_template.copy()
    sub_df[targets] = [
        test_preds_cat,
        test_preds_lgb,
        test_preds_xgb,
        test_preds_et,
        test_preds_rf,
        test_preds_lr
    ][idx]
    sub_df.to_csv(f"submissions/submission_{name}_v10.csv", index=False)
    sub_dfs.append(sub_df)

# Load metrics again for true labels alignment
y_true = metrics_df[targets].values
valid_mask = ~np.isnan(y_true)
num_models = len(raw_oof_preds)

# Apply Out-of-Fold Subject-Wise Beta Calibration
calibrated_oof_preds = [p.copy() for p in raw_oof_preds]
calibrated_sub_dfs = [df.copy() for df in sub_dfs]

print("Applying Out-of-Fold Subject-Wise Beta Calibration...")
for m_idx in range(num_models):
    for t_idx, target in enumerate(targets):
        mask = valid_mask[:, t_idx]
        raw_p_oof = raw_oof_preds[m_idx][:, t_idx]
        pred_mask = ~np.isnan(raw_p_oof)
        combined_mask = mask & pred_mask

        if combined_mask.sum() == 0:
            continue

        y_calib = y_true[combined_mask, t_idx]
        p_calib_raw = raw_p_oof[combined_mask]
        subjects = metrics_df.loc[combined_mask, id_col].values

        calib_p_oof = raw_p_oof.copy()
        calib_p_oof[combined_mask] = calibrate_beta_subject_wise(y_calib, p_calib_raw, subjects)
        calibrated_oof_preds[m_idx][:, t_idx] = calib_p_oof

        clf_global = calibrate_beta_global(y_calib, p_calib_raw)
        raw_p_test = sub_dfs[m_idx][target].values
        eps = 1e-15
        raw_p_test = np.clip(raw_p_test, eps, 1 - eps)
        logits_test = np.log(raw_p_test / (1 - raw_p_test))
        calibrated_sub_dfs[m_idx][target] = clf_global.predict_proba(logits_test.reshape(-1, 1))[:, 1]

print("✅ Out-of-fold calibration complete!")

In [ ]:
# =========================================================================
# 📝 submissions/submission_log.csv 자동 기록(Append) 및 고유 사본 저장 모듈
# =========================================================================
import datetime
import shutil

log_path = PROJECT_ROOT / "submissions" / "submission_log.csv" if 'PROJECT_ROOT' in locals() else Path("submissions/submission_log.csv")

if log_path.exists():
    log_df = pd.read_csv(log_path)
    new_id = log_df['id'].max() + 1
else:
    log_df = pd.DataFrame(columns=['id', 'date', 'submission_file', 'model_description', 'local_cv_score', 'public_lb_score', 'is_submitted', 'notes'])
    new_id = 1

current_time = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
mean_loss = np.mean(stacked_oof_overall)

# 1. Define highly unique file name to prevent future overwriting chaos!
unique_sub_name = f"submission_id_{new_id:03d}_score{mean_loss:.4f}.csv"
unique_sub_path = log_path.parent / unique_sub_name

# 2. Save final predictions directly to unique file name
stack_sub_df.to_csv(unique_sub_path, index=False)
print(f"📦 Saved unique SOTA prediction asset to: {unique_sub_path.name}")

# Also save to common file for safety
shutil.copy2(unique_sub_path, log_path.parent / "submission_calibrated_ensemble.csv")

# 3. Create new record row with unique file name mapped
new_row = {
    'id': new_id,
    'date': current_time,
    'submission_file': unique_sub_name,
    'model_description': f'SOTA-v21 Hybrid Stacking-Blending Calibrated Ensemble (LOSO) - CV {mean_loss:.6f}',
    'local_cv_score': round(mean_loss, 6),
    'public_lb_score': 0.0,
    'is_submitted': True,
    'notes': 'Executed automatically via SOTA legacy history pipeline Jupyter Notebook.'
}

# 4. Append and save
log_df = pd.concat([log_df, pd.DataFrame([new_row])], ignore_index=True)
log_df.to_csv(log_path, index=False)
print(f"📝 Successfully appended execution results to submission_log.csv under ID: {new_id}!")